# Synthetic → Plant Pipeline (Evidently Reports)

Este cuaderno toma **cada dataset sintético** como si fuera **una planta** real y ejecuta el **mismo flujo** que usas en producción: `run_drift_batch` → `make_report_for_plant` → Evidently HTMLs por estrategia.

**Plantas (sintéticas):**
- `planta1` → `synthetic_pervar_suddengrad_1min_3months.csv` (drifts mayormente *sudden*)
- `planta2` → `synthetic_pervar_gradualmix_1min_3months.csv` (drifts mayormente *gradual*)
- `planta3` → `synthetic_pervar_seasonal_1min_3months.csv` (estacional semanal)

Al final se muestra una **tabla-resumen** con las rutas de los HTML generados y una tabla de **errores** (si aparecen).


In [6]:
import pandas as pd, numpy as np
from pathlib import Path
import importlib.util

# Importa tus funciones reales
spec = importlib.util.spec_from_file_location("funciones_analisis", "../Analisis/Funciones_Drift.py")
funciones_analisis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(funciones_analisis)

# Rutas a datasets sintéticos originales
p_s = Path('synthetic_pervar_suddengrad_1min_3months.csv')
p_g = Path('synthetic_pervar_gradualmix_1min_3months.csv')
p_e = Path('synthetic_pervar_seasonal_1min_3months.csv')

# Creamos copias con columna 'date_time' (formato esperado por make_report_for_plant)
plants_dir = Path('synth_plants'); plants_dir.mkdir(parents=True, exist_ok=True)
def prepare_as_plant(src: Path, dst: Path):
    df = pd.read_csv(src, parse_dates=['timestamp'])
    if 'date_time' not in df.columns:
        df.insert(0, 'date_time', df['timestamp'])
    df.to_csv(dst, index=False)

prepare_as_plant(p_s, plants_dir/'planta1.csv')
prepare_as_plant(p_g, plants_dir/'planta2.csv')
prepare_as_plant(p_e, plants_dir/'planta3.csv')

# Mapeo planta -> archivo
plant_files = {
    'planta1': plants_dir/'planta1.csv',
    'planta2': plants_dir/'planta2.csv',
    'planta3': plants_dir/'planta3.csv',
}

plant_files

{'planta1': WindowsPath('synth_plants/planta1.csv'),
 'planta2': WindowsPath('synth_plants/planta2.csv'),
 'planta3': WindowsPath('synth_plants/planta3.csv')}

## Parámetros (idénticos a tu flujo) y estrategias

In [7]:
from typing import Dict, Tuple, Iterable, Optional, Literal

CURRENT_WINDOW = '3D'
RESAMPLE = None
RESAMPLE_AGG = 'mean'
EXCLUDE_COLUMNS = ['control_stable','var_variance']  # opcional excluir controles

# Detector primario (PSI) y umbral — puedes cambiar a 'auto' o ajustar
NUM_METHOD = 'psi'
NUM_THRESHOLD = 0.2

# Parámetros de referencia
DECAY_HALF_LIFE_HOURS = 24*7
DECAY_WEIGHT_MASS = 0.95

GOLDEN_WIN  = '30min'
GOLDEN_STEP = '10min'
GOLDEN_K    = 40

SEASONAL_WEEKS_BACK = 12

# Estrategias a ejecutar
strategies = ['decay','golden','seasonal']

# Directorio de salida de reportes HTML
OUTPUT_ROOT = Path('reports_synthetic')

## Ejecutar el batch (genera HTMLs Evidently por planta × estrategia)
Esto invoca internamente `make_report_for_plant(...)` para cada par (planta, estrategia) y guarda:
- `<OUTPUT_ROOT>/<planta>/<planta>_<estrategia>.html`
- `<OUTPUT_ROOT>/<planta>/<planta>_<estrategia>_metrics.csv` (tabla PSI)


In [8]:
paths, errors = funciones_analisis.run_drift_batch(
    plant_names=list(plant_files.keys()),
    strategies=strategies,
    plant_files=plant_files,
    flag_files={},  # sin flags en sintéticos
    output_root=OUTPUT_ROOT,
    CURRENT_WINDOW=CURRENT_WINDOW,
    RESAMPLE=RESAMPLE,
    RESAMPLE_AGG=RESAMPLE_AGG,
    EXCLUDE_COLUMNS=EXCLUDE_COLUMNS,
    NUM_METHOD=NUM_METHOD,
    NUM_THRESHOLD=NUM_THRESHOLD,
    DECAY_HALF_LIFE_HOURS=DECAY_HALF_LIFE_HOURS,
    DECAY_WEIGHT_MASS=DECAY_WEIGHT_MASS,
    GOLDEN_WIN=GOLDEN_WIN,
    GOLDEN_STEP=GOLDEN_STEP,
    GOLDEN_K=GOLDEN_K,
    SEASONAL_WEEKS_BACK=SEASONAL_WEEKS_BACK,
    SAVE_HTML=True  # <-- activar Evidently
)

# Resumen bonito
rows = []
for (pl, st), p in paths.items():
    rows.append({'planta': pl, 'estrategia': st, 'html_path': str(p)})
summary_ok = pd.DataFrame(rows).sort_values(['planta','estrategia'])

rows_e = []
for (pl, st), err in errors.items():
    rows_e.append({'planta': pl, 'estrategia': st, 'error': err})
summary_err = pd.DataFrame(rows_e).sort_values(['planta','estrategia']) if rows_e else pd.DataFrame()

display(summary_ok)
display(summary_err if not summary_err.empty else pd.DataFrame({'info':['Sin errores']}))

[OK] planta1 · decay → planta1_decay.html
[OK] planta1 · golden → planta1_golden.html
[OK] planta1 · seasonal → planta1_seasonal.html
[OK] planta2 · decay → planta2_decay.html
[OK] planta2 · golden → planta2_golden.html
[OK] planta2 · seasonal → planta2_seasonal.html
[OK] planta3 · decay → planta3_decay.html
[OK] planta3 · golden → planta3_golden.html
[OK] planta3 · seasonal → planta3_seasonal.html


,planta,estrategia,html_path
0,planta1,decay,reports_synthetic\planta1\planta1_decay.html
1,planta1,golden,reports_synthetic\planta1\planta1_golden.html
2,planta1,seasonal,reports_synthetic\planta1\planta1_seasonal.html
3,planta2,decay,reports_synthetic\planta2\planta2_decay.html
4,planta2,golden,reports_synthetic\planta2\planta2_golden.html
5,planta2,seasonal,reports_synthetic\planta2\planta2_seasonal.html
6,planta3,decay,reports_synthetic\planta3\planta3_decay.html
7,planta3,golden,reports_synthetic\planta3\planta3_golden.html
8,planta3,seasonal,reports_synthetic\planta3\planta3_seasonal.html


,info
0,Sin errores


# 📊 Evaluación Numérica de Desempeño

En esta sección se evalúa cuantitativamente el desempeño de las estrategias (`decay`, `golden`, `seasonal`) sobre las plantas sintéticas.
Se comparan las detecciones con las etiquetas reales diarias (`labels_long_*.csv`).

Se calculan las métricas:
- **TP**, **FP**, **TN**, **FN**
- **Accuracy**, **Precision**, **Recall**, **F1**, **FPR**
- **First Detection Delay (días)**

In [11]:
import pandas as pd
from pathlib import Path

# Rutas a etiquetas reales
LABELS = {
    'planta1': Path('labels_long_sudden_1D.csv'),
    'planta2': Path('labels_long_gradual_1D.csv'),
    'planta3': Path('labels_long_seasonal_1D.csv'),
}

def to_df(path):
    return pd.read_csv(path, parse_dates=['date_time'])

def day_ends_from_df(df, warmup_days=7):
    return pd.date_range(df['date_time'].min()+pd.Timedelta(days=warmup_days),
                         df['date_time'].max().normalize(), freq='1D')

def eval_run_for_plant(plant_name: str, df_path: Path):
    df_full = to_df(df_path)
    out_dir = OUTPUT_ROOT/plant_name
    out_dir.mkdir(parents=True, exist_ok=True)
    det_rows = []

    for strat in strategies:
        for wend in day_ends_from_df(df_full, warmup_days=7):
            df_cut = df_full[df_full['date_time'] <= (wend + pd.Timedelta(days=1) - pd.Timedelta(seconds=1))].copy()
            try:
                _ = funciones_analisis.make_report_for_plant(
                    df=df_cut, output_dir=out_dir, strategy=strat,
                    CURRENT_WINDOW=CURRENT_WINDOW, RESAMPLE=RESAMPLE, RESAMPLE_AGG=RESAMPLE_AGG,
                    EXCLUDE_COLUMNS=EXCLUDE_COLUMNS, NUM_METHOD=NUM_METHOD, NUM_THRESHOLD=NUM_THRESHOLD,
                    DECAY_HALF_LIFE_HOURS=DECAY_HALF_LIFE_HOURS, DECAY_WEIGHT_MASS=DECAY_WEIGHT_MASS,
                    GOLDEN_WIN=GOLDEN_WIN, GOLDEN_STEP=GOLDEN_STEP, GOLDEN_K=GOLDEN_K,
                    SEASONAL_WEEKS_BACK=SEASONAL_WEEKS_BACK,
                    plant_name=plant_name, flag_csv=None, SAVE_HTML=False
                )
                mpath = out_dir / f'{plant_name}_{strat}_metrics.csv'
                m = pd.read_csv(mpath)
                m_vars = m[m['col'].isin([f'var{i}' for i in range(1,5)])]
                drift_flag = int(m_vars['drift_detected'].fillna(False).any())
                det_rows.append({'plant':plant_name, 'strategy':strat,
                                 'window_start':wend, 'drift_flag':drift_flag})
            except Exception as e:
                det_rows.append({'plant':plant_name,'strategy':strat,
                                 'window_start':wend,'error':str(e)})
    return pd.DataFrame(det_rows)

def compute_metrics(detections_df: pd.DataFrame, labels_long: pd.DataFrame):
    label_agg = labels_long.groupby('window_start')['label_window_has_drift'].max().reset_index()
    out = []
    for (plant, strat), g in detections_df.groupby(['plant','strategy']):
        df = g.merge(label_agg, on='window_start', how='inner').rename(columns={'label_window_has_drift':'label'})
        tp = int(((df['drift_flag']==1)&(df['label']==1)).sum())
        fp = int(((df['drift_flag']==1)&(df['label']==0)).sum())
        tn = int(((df['drift_flag']==0)&(df['label']==0)).sum())
        fn = int(((df['drift_flag']==0)&(df['label']==1)).sum())
        acc = (tp+tn)/(tp+fp+tn+fn) if (tp+fp+tn+fn)>0 else None
        prec= tp/(tp+fp) if (tp+fp)>0 else None
        rec = tp/(tp+fn) if (tp+fn)>0 else None
        f1  = (2*prec*rec)/(prec+rec) if (prec and rec) else None
        fpr = fp/(fp+tn) if (fp+tn)>0 else None
        first_label = df.loc[df['label']==1,'window_start']
        first_detect= df.loc[df['drift_flag']==1,'window_start']
        delay = (first_detect.min()-first_label.min()).days if (len(first_label)>0 and len(first_detect)>0) else None
        out.append({'plant':plant,'strategy':strat,'TP':tp,'FP':fp,'TN':tn,'FN':fn,
                    'Accuracy':acc,'Precision':prec,'Recall':rec,'F1':f1,'FPR':fpr,
                    'First_Detection_Delay_days':delay})
    return pd.DataFrame(out)


In [12]:
# Ejecutar evaluación por planta
labels_map = {
    'planta1': pd.read_csv(LABELS['planta1'], parse_dates=['window_start']),
    'planta2': pd.read_csv(LABELS['planta2'], parse_dates=['window_start']),
    'planta3': pd.read_csv(LABELS['planta3'], parse_dates=['window_start']),
}

det_all = []
eval_tables = []

for plant_name, dfp in plant_files.items():
    det = eval_run_for_plant(plant_name, dfp)
    det_all.append(det)
    eval_tbl = compute_metrics(det, labels_map[plant_name])
    eval_tables.append(eval_tbl)

detections_df = pd.concat(det_all, ignore_index=True)
metrics_df = pd.concat(eval_tables, ignore_index=True).sort_values(['plant','strategy'])

print("=== Primeras filas de detecciones ===")
display(detections_df.head(10))

print("=== Métricas agregadas por planta y estrategia ===")
display(metrics_df)

=== Primeras filas de detecciones ===


,plant,strategy,window_start,drift_flag
0,planta1,decay,2025-01-08,0
1,planta1,decay,2025-01-09,0
2,planta1,decay,2025-01-10,0
3,planta1,decay,2025-01-11,0
4,planta1,decay,2025-01-12,0
5,planta1,decay,2025-01-13,0
6,planta1,decay,2025-01-14,0
7,planta1,decay,2025-01-15,0
8,planta1,decay,2025-01-16,0
9,planta1,decay,2025-01-17,0


=== Métricas agregadas por planta y estrategia ===


,plant,strategy,TP,FP,TN,FN,Accuracy,Precision,Recall,F1,FPR,First_Detection_Delay_days
0,planta1,decay,72,0,9,2,0.975904,1.000000,0.972973,0.986301,0.000000,2
1,planta1,golden,72,0,9,2,0.975904,1.000000,0.972973,0.986301,0.000000,2
2,planta1,seasonal,72,5,4,2,0.915663,0.935065,0.972973,0.953642,0.555556,-9
3,planta2,decay,49,0,29,5,0.939759,1.000000,0.907407,0.951456,0.000000,0
4,planta2,golden,54,0,29,0,1.000000,1.000000,1.000000,1.000000,0.000000,0
5,planta2,seasonal,53,3,26,1,0.951807,0.946429,0.981481,0.963636,0.103448,-29
6,planta3,decay,0,64,19,0,0.228916,0.000000,NaN,NaN,0.771084,None
7,planta3,golden,0,67,16,0,0.192771,0.000000,NaN,NaN,0.807229,None
8,planta3,seasonal,0,82,1,0,0.012048,0.000000,NaN,NaN,0.987952,None
